# Colab 05 — Validación externa (Montgomery, Shenzhen, JSRT)
Entrena el modelo original y el top 3 con el **mismo protocolo** y la misma configuración de búsqueda (sin nueva HPO) en cada dataset, y los evalúa en su 15 % retenido.
⚠️ Las máscaras son de **pulmón**, no de lesión: mide transferencia del método a otra anatomía y fuente, no segmentación de TB.
**Requisitos:** `datos/externos/<nombre>/{imgs,masks}` en Drive (`preparar_externos.py`), `top3.json` y `hpo/mejor_config.json`. Los datasets que falten se omiten.
5 semillas × 4 modelos × dataset; reanudable.

In [ ]:
# Celda 1 — GPU
import os
!nvidia-smi | head -15

In [ ]:
# Celda 2 — dependencias
%pip install -q segmentation-models-pytorch monai einops timm scipy scikit-image pandas opencv-python-headless tqdm pytest matplotlib onnx onnxruntime
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")
assert torch.cuda.is_available(), "Sin GPU: activa una GPU (Runtime → Change runtime type)"

In [ ]:
# Celda 3 — montar Drive y fijar rutas
from google.colab import drive
drive.mount('/content/drive')
CARPETA_DRIVE = "/content/drive/MyDrive/proyecto_tb"      # AJUSTAR si cambia
RAIZ_LOCAL = "/content/proyecto_tb"                        # copia local (rápida)
RES_HPO = f"{CARPETA_DRIVE}/resultados"                    # donde quedó la búsqueda (hpo/)
RES_E3 = f"{CARPETA_DRIVE}/resultados_e3"                  # resultados finales del Entregable 3
HPO_CFG = f"{RES_HPO}/hpo/mejor_config.json"
SPL = f"{RAIZ_LOCAL}/datos/splits"
os.makedirs(RES_E3, exist_ok=True)
assert os.path.isdir(f"{CARPETA_DRIVE}/codigo"), "Falta codigo/ en Drive"
assert os.path.isdir(f"{CARPETA_DRIVE}/datos/splits"), "Falta datos/splits"
print("Drive OK:", CARPETA_DRIVE, "\nResultados E3:", RES_E3)

In [ ]:
# Celda 4 — copiar código y datasets externos disponibles; generar NPZ + splits (70/15/15, semilla 2026)
import shutil, glob, json, subprocess, sys
shutil.copytree(f"{CARPETA_DRIVE}/codigo", f"{RAIZ_LOCAL}/codigo", dirs_exist_ok=True)
sys.path.insert(0, f"{RAIZ_LOCAL}/codigo")
DATASETS = []
for nombre in ("montgomery", "shenzhen", "jsrt"):
    origen = f"{CARPETA_DRIVE}/datos/externos/{nombre}"
    if not os.path.isdir(f"{origen}/imgs"):
        print("  (no está", nombre, "— se omite)"); continue
    shutil.copytree(origen, f"{RAIZ_LOCAL}/datos/externos/{nombre}", dirs_exist_ok=True)
    !cd "{RAIZ_LOCAL}" && python codigo/preparar_externos.py npz datos/externos/{nombre} --nombre {nombre}
    DATASETS.append(nombre)
print("datasets:", DATASETS)
top = json.load(open(f"{RES_E3}/top3.json"))["top"]
MODELOS = ["lacca_unet"] + top
MODELOS_STR = " ".join(MODELOS); TOP_STR = " ".join(top)
print("modelos:", MODELOS)

In [ ]:
# Celda 5 — ENTRENAR (repite hasta que todo diga "Nada pendiente")
for d in DATASETS:
    print("\n######", d)
    !python "{RAIZ_LOCAL}/codigo/driver_entrenamiento.py" --modelos {MODELOS_STR} --config-hpo "{HPO_CFG}" --splits-dir "{RAIZ_LOCAL}/datos/externos_splits/{d}" --resultados "{RES_E3}/externos/{d}" --max-minutos 150

In [ ]:
# Celda 6 — evaluar en el 15 % retenido de cada dataset
for d in DATASETS:
    print("\n######", d)
    !python "{RAIZ_LOCAL}/codigo/evaluar_externos.py" --resultados "{RES_E3}/externos/{d}" --splits-dir "{RAIZ_LOCAL}/datos/externos_splits/{d}" --modelos {MODELOS_STR}

In [ ]:
# Celda 7 — estadística por dataset (original vs top 3; Holm dentro de cada dataset)
for d in DATASETS:
    print("\n######", d)
    !python "{RAIZ_LOCAL}/codigo/estadistica_e3.py" --resultados "{RES_E3}/externos/{d}" --original lacca_unet --rivales {TOP_STR} --semillas 42 123 777 2024 31415 --archivo por_imagen_externo.csv --split ""